# ProteoWizardセットアップとインストール【論文再現シリーズ #3a】

## はじめに

この記事では、RAW→mzML変換に必要な **ProteoWizard** のインストールと環境設定を行います。現在のProteoWizardは従来とインストール先が変わっているため、最新の手順を詳しく解説します。

> **📝 INFO**
>
> **この記事で行う処理**
> ProteoWizard（オープンソースの質量分析データ変換ツール）をWindowsにインストールし、コマンドプロンプトから`msconvert`コマンドが使える状態にします。最新バージョンではAppDataフォルダにインストールされるため、正確な場所の特定と環境変数PATHの設定まで行います。

### 前提条件
- [#2b DIA-MSの理解](notebook_02b_data_formats.ipynb) が完了していること
- **Windows PC**（推奨）または Windows仮想マシン
- macOS Apple Siliconでは制約があります（記事末尾で説明）

In [ ]:
# 必要なライブラリをインポート
import os
import platform
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display, HTML, Image
import warnings
warnings.filterwarnings('ignore')

# システム情報を確認
print("🖥️ システム情報")
print(f"OS: {platform.system()} {platform.release()}")
print(f"アーキテクチャ: {platform.architecture()[0]}")
print(f"プロセッサ: {platform.processor()}")

# OS別の対応状況を表示
current_os = platform.system()
if current_os == "Windows":
    print("✅ ProteoWizard完全対応OS")
elif current_os == "Linux":
    print("⚠️ ProteoWizardはLinux対応版がありますが、Thermoライブラリは制限があります")
elif current_os == "Darwin":
    print("❌ macOSではWineまたは仮想マシンが必要です")
else:
    print("❓ 未知のOS")

## 🤔 なぜProteoWizardが必要なのか

**Thermo独自のRAWファイル**は以下の問題があります：

In [ ]:
# RAWファイルとmzMLファイルの比較
file_format_issues = {
    "問題点": [
        "ファイル形式",
        "読み込み可能なツール",
        "OS依存性",
        "長期保存性",
        "商用利用制限"
    ],
    "RAWファイルの課題": [
        "Thermo独自バイナリ形式",
        "Thermoツールのみ",
        "Windows専用ライブラリが必要",
        "将来の互換性に不安",
        "ライセンス制約"
    ],
    "mzMLの利点": [
        "国際標準のオープンフォーマット",
        "あらゆる解析ツール（sage、OpenMS等）",
        "OS非依存（Windows/Linux/macOS）",
        "10年後も確実に読める",
        "完全にオープン"
    ]
}

issues_df = pd.DataFrame(file_format_issues)
display(HTML(issues_df.to_html(index=False, escape=False)))

print("\n🎯 本シリーズでの変換の必要性:")
print("• sage-proteomics → mzML形式のみ対応")
print("• OpenSWATH → mzML形式のみ対応")
print("• 解析結果の再現性 → オープンフォーマットが重要")
print("• 長期アーカイブ → 国際標準形式が安全")

## 🛠️ ProteoWizardとは

**ProteoWizard** は、質量分析データの変換・処理を行うオープンソースツール群です。

In [ ]:
# ProteoWizardの基本情報
proteowizard_info = {
    "項目": [
        "開発元",
        "ライセンス",
        "主要ツール",
        "対応形式",
        "商用利用",
        "最新バージョン",
        "プラットフォーム"
    ],
    "内容": [
        "Vanderbilt大学 + Pacific Northwest National Laboratory",
        "Apache 2.0（商用利用可能）",
        "msconvert（RAW→mzML変換の事実上の標準）",
        "Thermo、Bruker、SCIEX、Waters、Agilent等全メーカー",
        "完全にOK（Apache 2.0ライセンス）",
        "3.0.26095+ (2024年版)",
        "Windows（メイン）、Linux（限定サポート）"
    ]
}

info_df = pd.DataFrame(proteowizard_info)
display(HTML(info_df.to_html(index=False, escape=False)))

print("\n🎯 ProteoWizardの主要機能:")
print("• msconvert → RAW/WIFF/d等からmzML/mzXML/MGF変換")
print("• msaccess → メタデータ・スペクトラム抽出")
print("• msdiff → ファイル比較・品質チェック")
print("• SeeMS → GUIベースのビューア")

## 💾 ProteoWizardのインストール

### Step 1: ダウンロードとインストール

1. [ProteoWizard 公式サイト](https://proteowizard.sourceforge.io/) にアクセス
2. **Download** → **Windows** から最新版をダウンロード
   ```
   pwiz-setup-3.0.XXXX-x86_64.msi (約500MB)
   ```
3. MSIファイルを実行してインストール

**インストール先について**:
- **記載上**: `C:\Program Files\ProteoWizard`
- **実際のデフォルト**: `C:\Users\[ユーザー名]\AppData\Local\Apps\ProteoWizard 3.0.XXXX...`

> **重要**: 最近のバージョンでは、管理者権限を必要としない **AppDataフォルダ** がデフォルトのインストール先になっています。

In [ ]:
# インストール先の候補リストを作成
def generate_installation_paths():
    """一般的なProteoWizardインストール先のリストを生成"""
    username = os.getenv('USERNAME', 'User')  # Windows
    if not username or username == 'User':
        username = os.getenv('USER', 'user')  # Unix系
    
    paths = [
        # 現在の標準的なインストール先（AppData）
        f"C:\\Users\\{username}\\AppData\\Local\\Apps\\ProteoWizard*",
        # 従来のインストール先
        "C:\\Program Files\\ProteoWizard",
        "C:\\Program Files (x86)\\ProteoWizard",
        # その他の可能性
        f"C:\\Users\\{username}\\AppData\\Local\\ProteoWizard*",
        "C:\\ProteoWizard",
    ]
    
    return paths

installation_paths = generate_installation_paths()

print("🔍 ProteoWizardの一般的なインストール先:")
for i, path in enumerate(installation_paths, 1):
    print(f"  {i}. {path}")

print("\n💡 インストール先検索のヒント:")
print("• 最新版 → AppDataフォルダ（管理者権限不要）")
print("• 古いバージョン → Program Filesフォルダ")
print("• カスタム → ユーザーが指定した任意の場所")

### Step 2: インストール場所の確認方法

**現在のProteoWizardは、デフォルトでAppDataフォルダにインストール**されます。以下の方法で場所を特定できます：

In [ ]:
# Windows向けの検索コマンドを生成
def generate_search_commands():
    """ProteoWizard検索用のコマンドを生成"""
    commands = {
        "コマンドプロンプト - 基本検索": [
            'dir "C:\\Users\\%USERNAME%\\AppData\\Local\\Apps\\ProteoWizard*"',
            'dir "C:\\Program Files\\ProteoWizard\\"',
            'dir "C:\\Program Files (x86)\\ProteoWizard\\"'
        ],
        
        "コマンドプロンプト - 全体検索（時間がかかる）": [
            'dir /s C:\\*msconvert.exe*',
            'dir /s "C:\\Users\\%USERNAME%\\*msconvert.exe*"',
            'dir /s "C:\\Program Files\\*msconvert.exe*"'
        ],
        
        "PowerShell - 高速検索": [
            'Get-ChildItem -Path C:\\ -Name "msconvert.exe" -Recurse -ErrorAction SilentlyContinue',
            'Get-ChildItem -Path C:\\ -Name "*ProteoWizard*" -Recurse -ErrorAction SilentlyContinue'
        ],
        
        "PATH確認（設定済みの場合）": [
            'where msconvert.exe',
            'msconvert --help'
        ]
    }
    
    return commands

search_commands = generate_search_commands()

print("🔍 ProteoWizard検索コマンド集")
print("=" * 50)

for method, commands in search_commands.items():
    print(f"\n📋 {method}:")
    for i, cmd in enumerate(commands, 1):
        print(f"  {i}. {cmd}")

print("\n💡 Windows検索機能を使う場合:")
print("1. Windowsキー → スタートメニュー")
print("2. 'msconvert' または 'ProteoWizard' で検索")
print("3. 見つかったアプリを右クリック → 'ファイルの場所を開く'")

print("\n⚠️ 注意事項:")
print("• AppDataフォルダは隠しフォルダ → エクスプローラーで '隠しファイル' を表示")
print("• 管理者権限不要 → AppDataがデフォルトインストール先")
print("• バージョン番号がパスに含まれる → 更新時にパスが変わる")

In [ ]:
# システム上でProteoWizardの存在を確認（Unix系システム用）
def check_proteowizard_unix():
    """Unix系システムでProteoWizardの有無を確認"""
    try:
        # msconvertコマンドの存在確認
        result = subprocess.run(['which', 'msconvert'], 
                              capture_output=True, text=True)
        if result.returncode == 0:
            return result.stdout.strip()
        else:
            return None
    except Exception:
        return None

# 現在のシステムでのProteoWizard確認
if platform.system() in ['Linux', 'Darwin']:
    pwiz_path = check_proteowizard_unix()
    if pwiz_path:
        print(f"✅ ProteoWizard検出: {pwiz_path}")
        # バージョン確認
        try:
            version_result = subprocess.run(['msconvert', '--help'], 
                                          capture_output=True, text=True)
            if 'Usage: msconvert' in version_result.stdout:
                print("✅ msconvertコマンド動作確認")
            else:
                print("⚠️ msconvertコマンドの動作に問題があります")
        except Exception:
            print("❌ msconvertコマンドの実行エラー")
    else:
        print("❌ ProteoWizardが見つかりません")
        if platform.system() == 'Darwin':
            print("💡 macOSでは以下の選択肢があります:")
            print("  1. Parallels/VMware等でWindows仮想マシン")
            print("  2. Wine経由での実行（不安定）")
            print("  3. Docker（Linux版ProteoWizard）")
        elif platform.system() == 'Linux':
            print("💡 LinuxではDocker版ProteoWizardが利用可能:")
            print("  docker pull proteowizard/wine")
elif platform.system() == 'Windows':
    print("🖥️ Windowsシステム検出")
    print("上記の検索コマンドでProteoWizardを探してください")
else:
    print(f"❓ 未知のOS: {platform.system()}")

## 🛤️ 環境変数PATHの設定

### Step 3: 環境変数PATHに追加

毎回長いパスを入力するのを避けるため、PATHに追加します：

In [ ]:
# PATH設定方法の比較
path_methods = {
    "方法": [
        "GUIで設定（推奨）",
        "PowerShellで設定",
        "setxコマンド",
        "一時的PATH設定"
    ],
    "難易度": [
        "易しい",
        "中級",
        "非推奨",
        "上級（一時的）"
    ],
    "永続性": [
        "永続",
        "永続",
        "永続（制限あり）",
        "セッションのみ"
    ],
    "特徴・注意点": [
        "文字数制限なし、確実",
        "スクリプト化可能",
        "1024文字制限で失敗しやすい",
        "コマンドプロンプト再起動で消失"
    ]
}

methods_df = pd.DataFrame(path_methods)
display(HTML(methods_df.to_html(index=False, escape=False)))

print("\n🎯 推奨方法: GUIで設定")
print("理由:")
print("• setxコマンドには1024文字の制限")
print("• 既存PATHが長い場合に新しいパスが切り捨てられる")
print("• GUIでは文字数制限がない")
print("• 確実に設定される")

In [ ]:
# PATH設定の具体的手順
print("🔧 PATH設定手順（Windows GUI）")
print("=" * 40)

gui_steps = [
    "1. 環境変数設定画面を開く",
    "   • Windowsキー + R → 'sysdm.cpl' → 詳細設定 → 環境変数",
    "   • または: rundll32 sysdm.cpl,EditEnvironmentVariables",
    "",
    "2. ユーザー環境変数の編集",
    "   • 'ユーザー環境変数' セクションで 'Path' を選択",
    "   • '編集' をクリック",
    "   • '新規' をクリック",
    "",
    "3. ProteoWizardパスを追加",
    "   例: C:\\Users\\[ユーザー名]\\AppData\\Local\\Apps\\ProteoWizard 3.0.26095.be91649 64-bit",
    "",
    "4. 保存",
    "   • OK → OK → OK で保存",
    "   • コマンドプロンプトを再起動"
]

for step in gui_steps:
    print(step)

print("\n💻 PowerShell版（コマンド派向け）")
print("=" * 40)
print("# PowerShellを管理者として実行")
print('$oldPath = [Environment]::GetEnvironmentVariable("Path", "User")')
print('$newPath = $oldPath + ";C:\\Users\\[ユーザー名]\\AppData\\Local\\Apps\\ProteoWizard..."')
print('[Environment]::SetEnvironmentVariable("Path", $newPath, "User")')

print("\n⚠️ 避けるべき方法")
print("=" * 20)
print('❌ setx PATH "%PATH%;..."')
print("理由: 1024文字制限でPATHが切り捨てられる")

### Step 4: 動作確認

**重要**: コマンドプロンプトを再起動してから実行

In [ ]:
# 動作確認用のテスト手順
print("✅ ProteoWizard動作確認手順")
print("=" * 40)

verification_steps = [
    "1. 新しいコマンドプロンプトを開く（重要！）",
    "2. 以下のコマンドを実行:",
    "   msconvert --help",
    "",
    "📋 成功時の出力例:",
    "Usage: msconvert [options] [filemasks]",
    "Convert mass spec data file formats.",
    "Return value: # of failed files.",
    "Options:",
    "  -f [ --filelist ] arg : specify text file containing filenames",
    "  -o [ --outdir ] arg   : set output directory ('-' for stdout)",
    "  -e [ --ext ] arg      : set extension for output files [mzML|mzXML|mgf...]",
    "  --mzML                : write mzML format [default]",
    "  ...",
    "",
    "🎯 確認ポイント:",
    "• 'msconvert' コマンドが認識される",
    "• 各種出力フォーマットが利用可能",
    "• '--mzML' がデフォルト形式"
]

for step in verification_steps:
    print(step)

print("\n❌ エラー時の対処法")
print("=" * 25)

error_solutions = [
    "エラー: 'msconvert' は、内部コマンドまたは外部コマンド...として認識されていません。",
    "",
    "🔧 対処法:",
    "1. PATH設定を再確認",
    "2. コマンドプロンプトを完全に再起動",
    "3. ProteoWizardの実際のインストール場所を再確認",
    "4. 管理者権限でコマンドプロンプトを実行",
    "5. システムを再起動（最後の手段）"
]

for solution in error_solutions:
    print(solution)

In [ ]:
# 典型的なインストール例の可視化
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

# インストールパターンの分布（架空のデータ）
install_patterns = {
    'AppData\\Local\\Apps': 65,  # 現在のデフォルト
    'Program Files': 20,        # 従来のデフォルト
    'Program Files (x86)': 10,  # 32bit版
    'カスタム': 5               # ユーザー指定
}

patterns = list(install_patterns.keys())
percentages = list(install_patterns.values())

colors = ['#2E8B57', '#4169E1', '#FF6347', '#DAA520']
wedges, texts, autotexts = ax.pie(percentages, labels=patterns, autopct='%1.1f%%',
                                  colors=colors, startangle=90)

ax.set_title('ProteoWizardインストール先の分布\n（2024年版の傾向）', 
             fontsize=14, fontweight='bold', pad=20)

# 凡例の追加
legend_labels = [
    'AppData (現在のデフォルト)',
    'Program Files (従来)',
    'Program Files x86 (32bit)',
    'カスタムパス'
]
ax.legend(wedges, legend_labels, title="インストール場所", 
          loc="center left", bbox_to_anchor=(1, 0, 0.5, 1))

plt.tight_layout()
plt.show()

print("📊 インストール先の変遷:")
print("• 2020年以前: Program Filesがデフォルト（管理者権限必要）")
print("• 2021年以降: AppDataがデフォルト（管理者権限不要）")
print("• 現在: AppData\\Local\\Apps\\が主流")
print("\n💡 検索のコツ: 新しい順番で場所を確認")

## 🍎 macOS Apple Siliconでの制約

macOSでのProteoWizard利用には制限があります：

In [ ]:
# macOS対応状況の整理
macos_compatibility = {
    "方法": [
        "ネイティブ実行",
        "Wine経由",
        "仮想マシン（Parallels）",
        "Docker（Linux版）",
        "クラウド（AWS/GCP）"
    ],
    "対応状況": [
        "❌ 非対応",
        "⚠️ 制限あり",
        "✅ 推奨",
        "🔶 部分対応",
        "✅ 完全対応"
    ],
    "詳細・注意点": [
        "macOS版ProteoWizardは存在しない",
        "Thermoライブラリが不安定、x86エミュレーション必要",
        "Windows仮想マシンで完全動作、ライセンス費用",
        "Thermoライブラリ制限、一部ファイル形式のみ", 
        "Windows環境を完全再現、処理能力が高い"
    ],
    "コスト": [
        "-",
        "無料（Wineは無料）",
        "中（Windowsライセンス）",
        "無料",
        "高（使用量課金）"
    ]
}

macos_df = pd.DataFrame(macos_compatibility)
display(HTML(macos_df.to_html(index=False, escape=False)))

print("\n🎯 macOS Apple Siliconユーザーへの推奨:")
print("\n🥇 最推奨: Parallels + Windows 11")
print("• 理由: 完全なWindows環境、Thermoライブラリ完全対応")
print("• コスト: Parallels $99/年 + Windows 11ライセンス")
print("• 性能: Apple Silicon M1/M2でも高速動作")

print("\n🥈 代替案: クラウド（AWS EC2 Windows）")
print("• 理由: 初期コストなし、高性能インスタンス利用可能")
print("• コスト: 使用時間課金（t3.medium ~$0.05/時間）")
print("• 利点: 大量ファイル処理時の高スペック利用")

print("\n🥉 実験的: Docker + Linux版 ProteoWizard")
print("• 理由: 無料、軽量")
print("• 制限: Thermoライブラリ制限、mzMLのみ出力")
print("• 用途: mzML→mzML変換、フィルタリング等")

if platform.system() == 'Darwin':
    print(f"\n🖥️ 現在のシステム: macOS {platform.mac_ver()[0]}")
    print("💡 本シリーズの推奨: すでにmzMLがある場合は変換をスキップ")

## 📋 まとめ

ProteoWizardのインストールと環境設定手順を確認しました。

In [ ]:
# セットアップ完了チェックリスト
checklist_items = {
    "項目": [
        "ProteoWizardダウンロード",
        "インストール実行",
        "インストール場所特定",
        "PATH環境変数設定",
        "コマンドプロンプト再起動",
        "msconvert動作確認",
        "次章への準備"
    ],
    "確認内容": [
        "公式サイトから最新版MSIファイルを取得",
        "MSIファイルを実行してインストール完了",
        "AppDataまたはProgram Filesから実際のパスを確認",
        "GUIまたはPowerShellでユーザーPATHに追加",
        "設定反映のため新しいコマンドプロンプトを開く",
        "'msconvert --help' でヘルプ画面が表示される",
        "変換対象のRAWファイルが準備されている"
    ],
    "ステータス": [
        "□",
        "□",
        "□", 
        "□",
        "□",
        "□",
        "□"
    ]
}

checklist_df = pd.DataFrame(checklist_items)
display(HTML(checklist_df.to_html(index=False, escape=False)))

print("\n🎯 重要ポイントの振り返り:")
print("• 最新版はAppDataフォルダにインストール（従来と異なる）")
print("• PATH設定はGUI推奨（setxコマンドは文字数制限あり）")
print("• コマンドプロンプト再起動が必須")
print("• macOSでは仮想マシン推奨")

print("\n🚀 次のステップ:")
print("✅ 全項目完了 → [#3b RAW→mzML変換実行](notebook_03b_convert_execution.ipynb)")
print("⚠️ すでにmzMLあり → [#4a Sage基礎](notebook_04a_sage_fundamentals.ipynb)")

print("\n#バイオインフォマティクス #プロテオミクス #ProteoWizard #環境設定 #labcode")